### Module 3

#### SQL Data Engineering and Inequality Gap

What this Module will Actually Solve :
Our raw data does not contain an "inequality gap" anywhere. 
It only contains individual rows one saying what the Female value was, a separate row saying what the Male value was 
for the same country, year, and indicator. 
    
Module 3's entire job is to find each matching Male/Female pair and subtract one from the other to produce a single, meaningful number: the gap.

In [1]:
import pandas as pd
import sqlite3

In [2]:
df = pd.read_csv('who_data_cleaned.csv')
conn = sqlite3.connect('who_capstone.db')
df.to_sql('health_data', conn, index=False, if_exists='replace')
cursor = conn.cursor()
cursor.execute('SELECT COUNT(*) FROM health_data')
print("Rows loaded into SQLite:", cursor.fetchone()[0])
conn.close()

Rows loaded into SQLite: 443150


In [3]:
conn = sqlite3.connect('who_capstone.db')
query = """
SELECT setting, date, subgroup, estimate
FROM health_data
WHERE dimension = 'Sex'
LIMIT 5
"""
df_results = pd.read_sql(query, conn)
df_results


,setting,date,subgroup,estimate
0,Albania,2002,Female,0.48
1,Albania,2002,Male,0.51
2,Albania,2005,Female,0.28
3,Albania,2005,Male,0.31
4,Albania,2008,Female,0.13


In [4]:
query = """
SELECT setting,COUNT(*) AS n_rows
FROM health_data
WHERE dimension = 'Sex'
GROUP BY setting
HAVING COUNT (*) > 1500
ORDER BY n_rows DESC
LIMIT 5
"""
pd.read_sql(query,conn)

,setting,n_rows
0,Spain,1800
1,Lithuania,1730
2,Latvia,1690
3,Portugal,1688
4,Finland,1662


In [5]:
query = """
SELECT subgroup,reference_subgroup,COUNT(*) as n
FROM health_data
WHERE dimension = 'Sex'
GROUP BY subgroup,reference_subgroup
"""
pd.read_sql(query,conn)

,subgroup,reference_subgroup,n
0,Female,0,80513
1,Male,1,80513


#### Adding an Index Before the Big Query

In [6]:
conn = sqlite3.connect('who_capstone.db')

cursor = conn.cursor()
cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_join_key
ON health_data(dimension, setting, date, indicator_abbr, reference_subgroup)
""")
conn.commit()
print('Index created')

Index created


A `JOIN` without an index on the join columns can be dramatically slower, especially as a table grows. 
Think of an index is like a book's table full of contents and without it, SQLite has to read every single page to find what it's looking for; 
with it, SQLite jumps straight to the right place.

#### Pairing Female Against the Male

In [10]:
query = """
SELECT a.setting,a.date,a.indicator_abbr,a.indicator_name,a.indicator_scale,a.whoreg6,a.subgroup AS female_label,a.estimate AS female_estimate,
b.subgroup AS male_label,b.estimate AS male_estimate,(a.estimate - b.estimate) AS inequality_gap,a.favourable_indicator,a.wbincome2025,a.iso3
FROM health_data a 
JOIN health_data b
ON a.setting = b.setting
AND a.date = b.date
AND a.indicator_abbr = b.indicator_abbr
AND a.dimension = b.dimension
WHERE a.dimension = 'Sex'
AND a.reference_subgroup = 0
AND b.reference_subgroup = 1
AND a.estimate IS NOT NULL
AND b.estimate IS NOT NULL
"""
gap_df = pd.read_sql(query,conn)
print('Total matched pairs:',len(gap_df))

Total matched pairs: 79448


Every one of the 79,448 output rows now represents one genuine, matched inequality observation — a specific country, in a specific year, for a specific health indicator, with a real number showing exactly how far apart Female and Male outcomes were.

#### Records on Each Side

In [13]:
query_counts = """
SELECT subgroup, COUNT(*) as n_with_valid_estimate
FROM health_data
WHERE dimension = 'Sex' AND estimate IS NOT NULL
GROUP BY subgroup
"""
pd.read_sql(query_counts,conn)

,subgroup,n_with_valid_estimate
0,Female,79953
1,Male,80008


In [14]:
query_unmatched_female = """
SELECT COUNT(*) as unmatched_female
FROM health_data a
LEFT JOIN health_data b
    ON a.setting = b.setting AND a.date = b.date
    AND a.indicator_abbr = b.indicator_abbr AND a.dimension = b.dimension
    AND b.reference_subgroup = 1 AND b.estimate IS NOT NULL
WHERE a.dimension = 'Sex' AND a.reference_subgroup = 0 AND a.estimate IS NOT NULL
    AND b.setting IS NULL
"""
print("\nUnmatched Female records:", pd.read_sql(query_unmatched_female, conn).iloc[0,0])

query_unmatched_male = """
SELECT COUNT(*) as unmatched_male
FROM health_data a
LEFT JOIN health_data b
    ON a.setting = b.setting AND a.date = b.date
    AND a.indicator_abbr = b.indicator_abbr AND a.dimension = b.dimension
    AND b.reference_subgroup = 0 AND b.estimate IS NOT NULL
WHERE a.dimension = 'Sex' AND a.reference_subgroup = 1 AND a.estimate IS NOT NULL
    AND b.setting IS NULL
"""
print("\nUnmatched Male records:", pd.read_sql(query_unmatched_male, conn).iloc[0,0])


Unmatched Female records: 505

Unmatched Male records: 560


"what did we lose, and how much?" 505 Female observations and 560 Male observations exist in the data but have no corresponding partner for the same country/year/indicator likely because that particular country simply never reported the other sex's value for that specific indicator in that specific year. 
This is normal, expected data sparsity, not an error.

In [15]:
print('Valid Female records: 79,953')
print('Matched pairs: 79,448')
print('Unmatched Female: ',79953 - 76448)
print('\nValid Male records: 80,008')
print('Matched pairs: 79,448')
print('Unmatched Male: ',80008 - 79448)

Valid Female records: 79,953
Matched pairs: 79,448
Unmatched Female:  3505

Valid Male records: 80,008
Matched pairs: 79,448
Unmatched Male:  560


In [16]:
duplicate_pairs = gap_df.duplicated(subset=['setting', 'date', 'indicator_abbr']).sum()
print("Duplicate pairs found:", duplicate_pairs)

Duplicate pairs found: 0


In [17]:
manual_check = df[(df['setting']=='Albania') & (df['date']==2002) &
                   (df['indicator_abbr']=='SI_POV_EMP1') & (df['dimension']=='Sex')]
print(manual_check[['setting','date','subgroup','estimate']])

female_val = manual_check[manual_check['subgroup']=='Female']['estimate'].values[0]
male_val = manual_check[manual_check['subgroup']=='Male']['estimate'].values[0]
manual_gap = female_val - male_val
print(f"\nManual calculation: {female_val} - {male_val} = {manual_gap}")

sql_result = gap_df[(gap_df['setting']=='Albania') & (gap_df['date']==2002) &
                     (gap_df['indicator_abbr']=='SI_POV_EMP1')]['inequality_gap'].values[0]
print(f"SQL result: {sql_result}")
print(f"Match: {abs(manual_gap - sql_result) < 0.0001}")

   setting  date subgroup  estimate
2  Albania  2002   Female      0.48
3  Albania  2002     Male      0.51

Manual calculation: 0.4799999892711639 - 0.5099999904632568 = -0.03000000119209295
SQL result: -0.03000000119209295
Match: True


#### Compare SQL Against an Independent Pandas Calculation

In [18]:
sex_df = df[df['dimension']=='Sex']
female_pd = sex_df[sex_df['reference_subgroup']==0][['setting','date','indicator_abbr','estimate']].rename(columns={'estimate':'female_est'})
male_pd = sex_df[sex_df['reference_subgroup']==1][['setting','date','indicator_abbr','estimate']].rename(columns={'estimate':'male_est'})

merged = female_pd.merge(male_pd, on=['setting','date','indicator_abbr'], how='inner')
merged = merged.dropna(subset=['female_est','male_est'])

print("Pandas merge result count:", len(merged))
print("SQL self-join result count:", len(gap_df))
print("Counts match:", len(merged) == len(gap_df))

Pandas merge result count: 79448
SQL self-join result count: 79448
Counts match: True


Two completely different tools, built with completely different code, arriving at the **exact same number** (79,448) is powerful evidence that this result reflects a genuine property of the data — not an artifact of a bug specific to one tool. This is the "compare SQL with Pandas"

#### Confirming Negative Gaps Are Present and Retained

In [19]:
neg_count = (gap_df['inequality_gap'] < 0).sum()
pos_count = (gap_df['inequality_gap'] > 0).sum()
zero_count = (gap_df['inequality_gap'] == 0).sum()

print(f"Negative gaps: {neg_count} ({neg_count/len(gap_df)*100:.1f}%)")
print(f"Positive gaps: {pos_count} ({pos_count/len(gap_df)*100:.1f}%)")
print(f"Zero gaps:     {zero_count} ({zero_count/len(gap_df)*100:.1f}%)")

Negative gaps: 49144 (61.9%)
Positive gaps: 27085 (34.1%)
Zero gaps:     3219 (4.1%)


"retained valid negative gaps rather than removing them automatically.Nearly 62% of all gaps are negative under our Female − Male convention, meaning in most country/indicator/year combinations, the Female value is lower than the Male value. This is a real, substantive finding, not noise to be cleaned away. No filtering, no `ABS()`, no removal happens here — every single one of these 79,448 rows, negative or positive, is preserved exactly as calculated.

In [20]:
gap_df.sample(5, random_state=42)[
    ['setting', 'date', 'indicator_name', 'female_estimate', 'male_estimate', 'inequality_gap']
]

,setting,date,indicator_name,female_estimate,male_estimate,inequality_gap
52506,Norway,2005,1.1.1 Population below international poverty l...,0.13208,0.29332,-0.16124
55656,Peru,2018,10.3.1 | 16.b.1 Population reporting having fe...,14.60000,15.00000,-0.40000
58613,Republic of Korea,2013,3.2.1 Under-five mortality rate (deaths per 10...,3.40000,4.00000,-0.60000
62276,Sao Tome and Principe,2014,4.1.2 Completion rate (%) - Primary education,85.53096,79.45765,6.07331
19543,Democratic Republic of the Congo,2016,3.2.1 Under-five mortality rate (deaths per 10...,85.80000,98.60000,-12.80000


shows 5 genuinely random rows from the final result, with `random_state=42` so this exact sample is reproducible if run it again.

In [22]:
gap_df.to_csv(r'inequality_gap_sex.csv', index=False)
print("Saved:", gap_df.shape)

Saved: (79448, 12)


In [23]:
conn.close()